# 실습 4 — 모델 학습과 해석

**가장 쉬운 방법 → 결과 → 문제 발견**

- **AutoGluon** — 여러 모델을 학습하고 비교하여 자동 선택
- 최적의 모델은 사전에 알 수 없음 → 다양하게 만들어 비교

### 이 노트북의 흐름

| | 내용 | |
|---|---|---|
| 1 | 가장 쉬운 방법 — 원본 + 랜덤 분할 + `fit()` 한 줄 | 자동 |
| 2 | `leaderboard()` — 성능 확인 | 자동 |
| 3 | **변수 중요도 — 무엇에 의존하는가** | **사람** ⚠️ |
| 4 | **제대로 다시 — 전처리 + 시간순 분할** | **사람** |
| 5 | **검증 성능과 평가 성능의 비교** | **사람** ⭐ |
| 6 | 학습 설정의 조정 | 사람 |
| 7 | 파운데이션 모델 | 자동 |
| 8 | 모델 저장 · 예측 | 자동 |

---

- ⏱ 표시 셀 — 학습이 돌아 시간 소요 (각 5분 내외)
- **실습 1~3을 건너뛰어도 단독 실행 가능**

---
## 0. 환경 준비 · 가장 쉬운 분할

- 원본 데이터를 그대로 사용 — 전처리 없이
- **무작위 80 : 20** — 가장 흔히 쓰는 방법

In [ ]:
import io, logging, warnings, contextlib
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor

warnings.filterwarnings("ignore")
logging.getLogger("autogluon").setLevel(logging.CRITICAL)
pd.set_option("display.max_columns", 50)

CSV     = "../SeoulBikeData.csv"     # 경로가 다르면 여기만 수정
LABEL   = "Rented Bike Count"
N_CPUS  = 16                       # 환경에 맞게 조정
LIMIT   = 300                      # 학습 시간 제한(초)
EXCLUDE = ["FASTAI"]              # 환경 오류 모델 제외

df = pd.read_csv(CSV, encoding="latin-1")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

# 가장 간단한 분할 — 무작위 80 : 20
train_data = df.sample(frac=0.8, random_state=42)
test_data  = df.drop(train_data.index)

print("학습용:", train_data.shape, "| 평가용:", test_data.shape)

---
## 1. `fit()` 한 줄로 학습

| 인자 | 역할 |
|---|---|
| `label` | 예측 대상 컬럼명 |
| `eval_metric` | 성능 판단 기준 (`r2` · `rmse` · `mae` …) |
| `path` | 모델 저장 경로 |
| `time_limit` | 학습 최대 시간(초) |

### 로그에서 확인할 점

- **문제 유형 자동 판별** — 대여량이 숫자 → 회귀(regression)
- **전처리 자동 수행** — 원본 열 → 처리된 특징 *(실습 2)*
- **검증용 자동 분할** — `holdout_frac=0.1` · 학습용의 10%
- **모델을 차례로 학습** — 각 모델 옆에 검증 점수
- **최종 모델 자동 선택** — `Best model` · 대개 가중 앙상블

In [ ]:
# ⏱ 학습 — 약 5분
predictor = TabularPredictor(
    label=LABEL,
    eval_metric="r2",
    path="ag_easy",
).fit(
    train_data,
    time_limit=LIMIT,
    ag_args_fit={"num_cpus": N_CPUS},
    excluded_model_types=EXCLUDE,
)

> **`fit()`에 넣는 것은 학습용만** — 평가용은 넣지 않음
> 검증용은 AutoGluon이 학습용을 다시 쪼개 **자동 생성** *(실습 3)*

---
## 2. `leaderboard()` — 모델별 성능 순위

| 열 | 의미 |
|---|---|
| `model` | 모델 이름 |
| `score_test` | **평가 성능** — 학습에 쓰지 않은 데이터 |
| `score_val` | **검증 성능** — AutoGluon이 자동 분할한 데이터 |
| `fit_time` | 학습 소요 시간(초) |

- `eval_metric="r2"` → **1에 가까울수록 좋음**

In [ ]:
lb_easy = predictor.leaderboard(test_data, silent=True)
lb_easy[["model", "score_test", "score_val", "fit_time"]]

#### 확인 사항

- R² 0.9 이상 — **높은 성능**
- 코드 세 줄로 여기까지

> **이 숫자를 신뢰할 수 있는가** — 다음 장에서 확인

---
## 3. 변수 중요도 — 무엇에 의존하는가 ⚠️

### 순열 중요도 (Permutation Importance) 원리

1. 정상 예측 — 원본으로 예측해 오차 측정
2. 한 변수 뒤섞기 — 그 열만 행끼리 무작위로 섞어 정답과의 관계를 끊고 재예측
3. **하락폭 = 중요도**

| 하락폭이 클수록 | 중요한 변수 — 없으면 예측이 크게 망가짐 |
|---|---|
| 하락폭 ≈ 0 | 예측에 거의 기여하지 않음 |
| **음수(−)** | 오히려 해로운 변수 — 제거 시 성능 향상 가능 |

> "이 변수가 없으면 예측이 얼마나 나빠질까"를 직접 측정

In [ ]:
# ⏱ 순열 중요도 — 변수마다 반복 예측 (수 분)
imp = predictor.feature_importance(test_data)
imp[["importance", "stddev", "p_value"]].head(8)

#### 상위 변수 점검

| 확인 질문 | 해당되면 |
|---|---|
| 예측 시점에 알 수 있는 값인가 | 아니면 → **누수** |
| 예측 대상과 정의상 연결된 변수인가 | 그렇다면 → **누수** |
| 시간·ID 관련 변수가 상위인가 | **분할 구조 재검토** |

In [ ]:
# Functioning Day 점검 — 대여량과의 관계
z  = (df[LABEL] == 0)
nf = (df["Functioning Day"] == "No")

print("대여량 = 0          :", z.sum(), "건")
print("Functioning Day = No:", nf.sum(), "건")
print("둘 다 해당          :", (z & nf).sum(), "건  ← 완전히 일치")
print()
print("→ 운영 중단일은 예외 없이 대여량 0")
print("→ 모델이 이 규칙에 의존 — 연구자가 이미 아는 것")

> ⚠️ **중요도 상위 = 좋은 변수가 아님**
> 중요도 = **모델이 무엇에 의존하는가** ≠ **무엇이 진짜 원인인가**

### 발견한 두 가지 문제

| ① 데이터 | `Functioning Day` — 답을 이미 아는 열 · 예측하려는 질문이 아님 |
|---|---|
| ② 분할 | 시계열인데 **무작위로 나눔** — 6월 3일 14시·16시로 학습하고 15시를 평가 |

> **다음 장에서 둘 다 수정하여 재학습**

---
## 4. 제대로 다시 — 전처리 + 시간순 분할

- 실습 2의 전처리 — 위장 결측 · 누수 행 제거
- 실습 3의 분할 — **날짜 경계에서** 시간순으로

In [ ]:
def prepare(path=CSV, ratio=0.8):
    """실습 2·3의 핵심을 한 함수로"""
    x = pd.read_csv(path, encoding="latin-1")
    x["Date"] = pd.to_datetime(x["Date"], format="%d/%m/%Y")   # ① 날짜 인식
    x.loc[x["Humidity(%)"] == 0, "Humidity(%)"] = np.nan       # ② 위장 결측
    x = (x[x["Functioning Day"] == "Yes"]                      # ③ 누수 열·행 제거
           .drop(columns=["Functioning Day"])
           .sort_values(["Date", "Hour"])
           .reset_index(drop=True))

    days = np.sort(x["Date"].unique())        # ④ 날짜 경계에서 자름
    cut  = days[int(len(days) * ratio)]       #    같은 날이 양쪽에 걸치지 않도록
    return x[x["Date"] < cut], x[x["Date"] >= cut]


train_ok, test_ok = prepare()

print("학습용:", train_ok.shape, "| 평가용:", test_ok.shape)
print("학습 기간:", train_ok["Date"].min().date(), "~", train_ok["Date"].max().date())
print("평가 기간:", test_ok["Date"].min().date(),  "~", test_ok["Date"].max().date())
print("겹치는 날짜:", len(set(train_ok["Date"]) & set(test_ok["Date"])), "일")

In [ ]:
# ⏱ 다시 학습 — 약 5분
predictor_ok = TabularPredictor(
    label=LABEL, eval_metric="r2", path="ag_ok",
).fit(
    train_ok, time_limit=LIMIT,
    ag_args_fit={"num_cpus": N_CPUS},
    excluded_model_types=EXCLUDE,
)

In [ ]:
from sklearn.metrics import mean_absolute_error

lb_ok = predictor_ok.leaderboard(test_ok, silent=True)

rows = []
for name, p, te, lb in [("원본 + 랜덤 분할",     predictor,    test_data, lb_easy),
                        ("전처리 + 시간순 분할", predictor_ok, test_ok,   lb_ok)]:
    mae = mean_absolute_error(te[LABEL], p.predict(te))
    rows.append((name, lb["score_test"].max(), mae, mae/te[LABEL].mean()*100))

print("=" * 62)
print("%-24s %8s %10s %12s" % ("", "R²", "평균 오차", "오차/평균"))
print("-" * 62)
for n, r, m, p in rows:
    print("%-24s %8.3f %8.0f대 %11.1f%%" % (n, r, m, p))
print("=" * 62)

In [ ]:
# ⏱ 순열 중요도 — 변수마다 반복 예측 (수 분)
imp = predictor.feature_importance(ㅅㄷㄴㅅ)
imp[["importance", "stddev", "p_value"]].head(8)

#### 결과 해석

- 성능이 **하락** — 그러나 이쪽이 **현실적인 수치**
- 앞의 높은 점수는 누수와 사이 채우기의 결과

> ⚠️ **평가 조건이 달라진 점도 고려**
> 시간순 평가 기간 = 가을 · **학습에 없던 계절**
> 격차에는 누수 제거 효과와 **계절 외삽의 어려움**이 함께 포함
> → 실무에서 마주하는 것이 정확히 이 상황 — **미래는 늘 학습에 없던 시기**

---
## 5. 검증 성능과 평가 성능의 비교 ⭐

**AutoGluon의 점수도 확인이 필요**

- 시간순으로 나눠 학습해도 — **AutoGluon의 검증용은 학습 기간 내부에서 추출**
- 이 데이터가 시계열이라는 것을 모름
- → `score_val`이 `score_test`보다 높게 나옴

In [ ]:
cmp = lb_ok[["model", "score_val", "score_test"]].copy()
cmp["하락폭"] = (cmp["score_val"] - cmp["score_test"]).round(3)
print(cmp.sort_values("score_test", ascending=False)
         .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()

by_val  = lb_ok.sort_values("score_val",  ascending=False).iloc[0]
by_test = lb_ok.sort_values("score_test", ascending=False).iloc[0]

print("검증 성능 1위 :", by_val["model"],
      "(검증 %.3f → 평가 %.3f)" % (by_val["score_val"], by_val["score_test"]))
print("평가 성능 1위 :", by_test["model"], "(평가 %.3f)" % by_test["score_test"])
print()
print("하락폭 — 최소 %.3f · 최대 %.3f" % (cmp["하락폭"].min(), cmp["하락폭"].max()))

#### 확인되는 사실

- **예외 없이 하락** — 모든 모델
- **순위 역전** — 검증 기준 순위와 평가 기준 순위가 다름
- **하락폭이 일정하지 않음** — 어느 모델이 더 하락할지 사전에 알 수 없음

### 원인 — 두 가지가 겹침

| **① 검증 성능이 낙관적** | 검증용이 **학습 기간 내부**에서 추출<br>같은 시기의 이웃 데이터가 학습에 포함 |
|---|---|
| **② 평가 조건이 다름** | 평가 기간 = 2018.09 ~ 11 · **가을만**<br>학습에 없던 시기 |

- 하락폭이 모델마다 다른 점 → **②의 영향**
- 검증셋이 같으므로 ①만 원인이라면 하락폭도 비슷해야 함

> **두 원인의 분리에는 별도 실험 필요**
> 요점 — **보고된 점수를 그대로 신뢰할 수 없음**

> **잘못된 검증 → 잘못된 판단**
> AutoML은 주어진 기준 안에서 최선을 선택 · **기준이 틀리면 판단도 틀림**
> 분할 구조를 정하는 것은 **연구자의 역할**

---
## 6. 학습 설정의 조정

**주요 하이퍼파라미터**

| 옵션 | 역할 | 값 |
|---|---|---|
| `presets` | 성능 ↔ 속도 | `medium` *(기본)* → `best` → `extreme` |
| `time_limit` | 학습 최대 시간(초) | 600 (10분) |
| `eval_metric` | 성능 판단 기준 | 회귀 `r2` · `rmse` / 분류 `accuracy` · `f1` |
| `num_bag_folds` | 교차검증 조각 수 | 5 *(실습 3)* |

### presets — AutoGluon 1.5

| `medium` | 빠른 학습 — **지정하지 않으면 이것** |
|---|---|
| `best` | 정확도 최대 |
| `extreme` | **파운데이션 모델 포함** — GPU 필요 *(7장)* |

> 로그의 경고 — `No presets specified! ... Defaulting to 'medium'`
> 지금까지의 실습은 모두 `medium` — 프로토타이핑용 기본값

In [ ]:
# ⏱ 정확도 우선 설정 — 시간이 오래 걸림 (10분+) · 필요 시 주석 해제
# predictor_best = TabularPredictor(
#     label=LABEL, eval_metric="r2", path="ag_best",
# ).fit(
#     train_ok,
#     presets="best",               # 정확도 우선
#     time_limit=600,               # 10분 제한
#     num_bag_folds=5,              # 교차검증
#     ag_args_fit={"num_cpus": N_CPUS},
#     excluded_model_types=EXCLUDE,
# )
# predictor_best.leaderboard(test_ok, silent=True).head()

> ⚠️ **설정을 바꿔가며 평가용 점수로 선택하면 — 평가용에 맞춘 것**
> 설정 선택은 **검증용**(`score_val`)으로
> 평가용은 **결과를 보고 되돌아가지 않기** *(실습 3)*

---
## 7. 표 데이터용 파운데이션 모델

| **일반 모델** | 내 데이터로 처음부터 학습 |
|---|---|
| **파운데이션 모델** | 대량의 표 데이터로 **사전학습 완료** · 그대로 활용 |

- 추가 학습 없이 다양한 작업에 적용
- **소량 데이터에서 특히 유리** — 사전학습 지식 활용
- AutoGluon 1.5 내장 — TabPFNv2 · TabICL · Mitra · TabDPT · TabM

> ⚠️ **사전 준비 필요**
> `pip install autogluon.tabular[tabarena]` · **GPU 필요**

In [ ]:
# ⏱ ① preset으로 — 파운데이션 모델 포함 전체 구성 · 필요 시 주석 해제
# predictor_fm = TabularPredictor(
#     label=LABEL, eval_metric="r2", path="ag_extreme",
# ).fit(train_ok, presets="extreme", time_limit=600,
#        ag_args_fit={"num_cpus": N_CPUS})

# ⏱ ② 직접 지정 — 파운데이션 모델만
# predictor_fm = TabularPredictor(
#     label=LABEL, eval_metric="r2", path="ag_fm",
# ).fit(train_ok, hyperparameters={"TABPFNV2": {}, "MITRA": {}},
#        ag_args_fit={"num_cpus": N_CPUS})

# predictor_fm.leaderboard(test_ok, silent=True)[["model","score_test","fit_time"]]

> **학습 시간 대비 성능을 함께 볼 것**
> 파운데이션 모델 — 수 초 만에 앙상블에 근접하는 경우 있음
> `extreme` — 10만 건 미만 데이터에서 `best` 보다 크게 우수 *(AutoGluon 문서)*

> 데이터가 수백 행이라면 — 교차검증 *(실습 3)* · **파운데이션 모델** 둘 다 검토

---
## 8. 모델 저장 · 예측

- `fit()` 실행 시 `path`에 자동 저장됨
- 새 세션·다른 PC에서 `load()`로 불러와 사용

In [ ]:
# 새 데이터 예측 — 정답 열이 없는 데이터
new_data = test_ok.drop(columns=[LABEL]).head(5)

result = new_data[["Date", "Hour", "Temperature(°C)", "Seasons"]].copy()
result["예측 대여량"] = predictor_ok.predict(new_data).round(1)
result["실제 대여량"] = test_ok[LABEL].head(5).values
result

In [ ]:
# 저장된 모델 불러오기
loaded = TabularPredictor.load("ag_ok")
print("불러오기 완료 —", loaded.model_best)
print("예측 확인:", loaded.predict(new_data).round(1).tolist())

---
## 정리

| AutoGluon이 알아서 | 연구자가 확인해야 |
|---|---|
| 문제 유형 판별 (회귀 · 분류) | **분할 구조** — 무엇이 묶여 있는가 |
| 여러 모델 학습 · 앙상블 | **`score_val` vs `score_test`** — 하락폭 |
| 최종 모델 선택 | 그 선택이 평가 기준으로도 최고인가 |
| 검증용 자동 분할 (학습용의 10%) | 설정 선택은 검증용 · 평가용은 최종 보고 |
| 변수 중요도 계산 | **상위 변수가 누수인가** |

### 이 노트북의 흐름

| ① | 가장 쉬운 방법 — 코드 세 줄로 R² 0.9 이상 |
|---|---|
| ② | 변수 중요도 → **누수 발견** |
| ③ | 전처리 + 시간순 분할로 재학습 → **성능 하락** |
| ④ | 그마저도 검증용은 학습 기간 내부 → **`score_val`도 낙관적** |

---

## **AutoML은 주어진 기준 안에서 최선을 선택한다**

> 기준을 정하는 것은 연구자
> 무엇을 주는가 · 어떻게 나누는가 · 결과를 어떻게 읽는가